In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

import torch

if torch.cuda.is_available():
    # GPU is available
    print("GPU is available")
else:
    # GPU is not available
    print("No GPU available, using CPU")

## Text Ranking

In [ ]:
%%capture
!pip install --upgrade scipy networkx
!pip install scipy==1.8.0
!pip install networkx==2.6.3
!pip3 install transformers
!pip3 install sentence_transformers

import nltk
from nltk.cluster.util import cosine_distance
import pandas as pd
import networkx as nx
import re
import numpy as np
import IPython
from IPython.display import display,HTML
import operator
import time
import math
import regex
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import torch
from sklearn.metrics.pairwise import cosine_similarity

import torch
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
%%capture
tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/banglabert_generator')
model = AutoModel.from_pretrained('csebuetnlp/banglabert_generator')
model.to(device)

In [ ]:
def bn_word_similarity(doc1, doc2):
  ###Tokenize the words like before:
  doc = []
  doc.append(doc1)
  doc.append(doc2)

  # initialize dictionary: stores tokenized words
  token = {'input_ids': [], 'attention_mask': []}
  for word in doc:
      # encode each word, append to dictionary
      new_token = tokenizer.encode_plus(word, max_length=400, #min_length=64,
                                        truncation=True, padding='max_length',
                                        return_tensors='pt')
      token['input_ids'].append(new_token['input_ids'][0])
      token['attention_mask'].append(new_token['attention_mask'][0])
  # reformat list of tensors to single tensor
  token['input_ids'] = torch.stack(token['input_ids'])
  token['attention_mask'] = torch.stack(token['attention_mask'])

  #Process tokens through model:
  # token.to(device)
  token = {k: v.to(device=device, non_blocking=True) for k, v in token.items()}
  output = model(**token)
  # output = model(**token).cuda()
  #output.keys()

  #The dense vector representations of text are contained within the outputs 'last_hidden_state' tensor
  embeddings = output.last_hidden_state
  #embeddings

  # To perform this operation, we first resize our attention_mask tensor:
  att_mask = token['attention_mask']
  #att_mask.shape

  mask = att_mask.unsqueeze(-1).expand(embeddings.size()).float()
  #mask.shape

  mask_embeddings = embeddings * mask
  #mask_embeddings.shape

  #Then we sum the remained of the embeddings along axis 1:
  summed = torch.sum(mask_embeddings, 1)
  #summed.shape

  #Then sum the number of values that must be given attention in each position of the tensor:
  summed_mask = torch.clamp(mask.sum(1), min=1e-9)
  #summed_mask.shape

  mean_pooled = summed / summed_mask
  #mean_pooled

  #Let's calculate cosine similarity for word 0:
  # convert from PyTorch tensor to numpy array
  mean_pooled = mean_pooled.cpu().detach().numpy()
  # calculate
  return cosine_similarity([mean_pooled[0]],mean_pooled[1:])[0][0]

def bn_build_similarity_matrix(words):
    # Create an empty similarity matrix
    similarity_matrix = np.zeros((len(words), len(words)))

    for idx1 in range(0, len(words)):
            # print(idx1)
            for idx2 in  range(0, len(words)):
                similarity_matrix[idx1][idx2] = bn_word_similarity(words[idx1], words[idx2])

    return similarity_matrix


def bn_generate_keywords(file_name, top_n=5):
    global text2GenKeys
    text2GenKeys = []

    # Step 1 - Read texts
    words = file_name

    # Step 2 - Generate Similary Martix across words
    word_similarity_martix = bn_build_similarity_matrix(words)

    # Step 3 - Rank words in similarity martix
    word_similarity_graph = nx.from_numpy_array(word_similarity_martix)
    scores = nx.pagerank(word_similarity_graph)

    # Step 4 - Sort the rank and pick top words
    ranked_word = sorted(((scores[i],s) for i,s in enumerate(words)), reverse=True)
    # print("Indexes of top ranked_word order are ", ranked_word)

    for i in range(top_n):
      text2GenKeys.append("".join(ranked_word[i][1]))

    # Step 5 - Offcourse, output the text2GenKeys
    return text2GenKeys

In [ ]:
def textrank(text):
    rank = bn_generate_keywords((text.replace('।', '')).split(), int(len((text.replace('।', '')).split())*0.6))
    return rank

In [ ]:
text = 'গত ২০ বছরে তিনি রাশিয়ার প্রেসিডেন্ট এবং প্রধানমন্ত্রী হিসেবে দায়িত্ব পালন করেছেন।'
textrank(text)

['প্রধানমন্ত্রী', 'তিনি', 'প্রেসিডেন্ট', 'পালন', 'করেছেন', 'হিসেবে', 'এবং']

## LIAAD/yake

In [ ]:
%%capture
!pip install git+https://github.com/LIAAD/yake

In [ ]:
import yake

language = "bn"
max_ngram_size = 1
deduplication_threshold = 0.9
deduplication_algo = 'seqm'
windowSize = 1
numOfKeywords = 20

custom_kw_extractor = yake.KeywordExtractor(lan=language, n=max_ngram_size, dedupLim=deduplication_threshold, dedupFunc=deduplication_algo, windowsSize=windowSize, top=numOfKeywords, features=None)
def remove_dari(input_string):
    return input_string.replace('।', '')

def LIAADyake(text):
    keywords = custom_kw_extractor.extract_keywords(text)
    keyslist = []
    for kw in keywords:
        keyslist.append(remove_dari(kw[0]))
#     return ' '.join(keyslist)
    return keyslist

sen ='এরপর গ্রেপ্তার করা হয় আরেক শিক্ষার্থীকে।'
LIAADyake(sen)

['এরপর', 'শিক্ষার্থীকে', 'গ্রেপ্তার', 'করা', 'আরেক']

## bn Key2Text sagorsarker/bangla-bert-base

In [ ]:
%%capture
!pip install transformers torchvision
!pip install git+https://github.com/csebuetnlp/normalizer

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import requests
import pandas as pd
import numpy as np
import random
import string

stop_wrds = ["!","\"","‘","’","#",".","$","%","&","\'","(",")","*","+",",", "-",".","/",":",";","<","=",">","?","@","[","\\","\]","^","_","`","{","|","}","~","[CLS]","[PAD]","[SEP]","[UNK]","।"]

class KeywordExtractorBERTBASE:
    def __init__(self):
        self.tokenizer = None
        self.model = None

    def load_model(self):
        # model_name = 'csebuetnlp/banglabert'
        model_name = 'sagorsarker/bangla-bert-base'
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()

    def score_words(self, sentence):
        input_ids = torch.tensor([self.tokenizer.encode(sentence, max_length=512, padding='max_length', add_special_tokens=True, truncation=True)])
        # input_ids = torch.tensor([self.tokenizer.encode(sentence)])
        tokenized_text = self.tokenizer.convert_ids_to_tokens(input_ids[0])

        with torch.no_grad():
            outputs = self.model(input_ids=input_ids)

        embeddings = outputs.last_hidden_state.squeeze(0)
        mean_embedding = embeddings.mean(dim=0)

        original_words = []
        original_word_embeddings = []
        current_word = ""
        prev_word = ""
        prev_embedding = ""
        current_embedding_lst = []

        for i, embedding in enumerate(embeddings):

            token = tokenized_text[i]
            if token in stop_wrds:
              continue
            elif token.startswith("##"):
                current_word = prev_word + token[2:]
                current_embedding_lst.append(prev_embedding)
                current_embedding_lst.append(embedding)
                current_embedding = sum(current_embedding_lst)/len(current_embedding_lst)

                original_words.pop()
                original_word_embeddings.pop()

                original_words.append(current_word)
                original_word_embeddings.append(current_embedding)

                prev_word = current_word
                prev_embedding = current_embedding
                current_embedding_lst = []
            else:
                original_words.append(token)
                original_word_embeddings.append(embedding)
                prev_word = token
                prev_embedding = embedding

        # for i, (token, embedding) in enumerate(zip(original_words, original_word_embeddings)):
        #   print(token)
        #   print(embedding.shape)

        word_scores = []
        new_scores = []
        for i in range(len(original_word_embeddings)):
            cos_sim = torch.nn.functional.cosine_similarity(original_word_embeddings[i], mean_embedding, dim=0)
            word_scores.append((original_words[i], cos_sim.item()))
            new_scores.append((original_words[i], original_word_embeddings[i], cos_sim.item()))

        return word_scores, mean_embedding, new_scores

    def print_top_values(self, data):
        top_values = int(len(data) * 0.6)
        if top_values < 10:
            top_values = int(len(data) * 0.7)
        if top_values < 4:
            top_values = int(len(data) * 0.8)

        finalLst = []
        for i in range(top_values):
            finalLst.append(data[i][0])

        return finalLst

    def keysOfSentence(self, sentence):
        final = []
        word_scores, mean_embedding, new_scores = self.score_words(sentence)
        word_scores.sort(key = lambda x: x[1], reverse=True)
        new_scores.sort(key = lambda x: x[2], reverse=True)

        finalKeysLst = self.print_top_values(word_scores)
        finalKeys = finalKeysLst

        return finalKeys, mean_embedding, new_scores

    def extract_keywords(self, text):
        self.load_model()
        return self.keysOfSentence(text)

def keysOfSentenceBERTBASE(text):
    extractor = KeywordExtractorBERTBASE()

    # Load the model and tokenizer
    extractor.load_model()

    # Use GPU if available
    if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    extractor.model.to(device)
    extractor.model.eval()

    keywords, mean_embedding, new_scores = extractor.extract_keywords(text)
    return keywords

In [ ]:
text = 'এরপর গ্রেপ্তার করা হয় আরেক শিক্ষার্থীকে।'
keysOfSentenceBERTBASE(text)

vocab.txt: 100%|██████████| 2.24M/2.24M [00:04<00:00, 470kB/s]
C:\Users\USER\anaconda3\envs\pytorchGPU\lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
config.json: 100%|██████████| 491/491 [00:00<?, ?B/s] 
model.safetensors

['গরেপতার', 'শিকষারথীকে', 'আরেক', 'হয']

## bn Key2Text Main

In [ ]:
import torch
import torch.nn.functional as F
from transformers import ElectraTokenizer, ElectraModel
import requests
import pandas as pd
import numpy as np
import random
import string

stop_wrds = ["!","\"","‘","’","#",".","$","%","&","\'","(",")","*","+",",", "-",".","/",":",";","<","=",">","?","@","[","\\","\]","^","_","`","{","|","}","~","[CLS]","[PAD]","[SEP]","[UNK]","।"]

class KeywordExtractor:
    def __init__(self):
        self.tokenizer = None
        self.model = None

    def load_model(self):
        model_name = 'csebuetnlp/banglabert'
        self.tokenizer = ElectraTokenizer.from_pretrained(model_name)
        self.model = ElectraModel.from_pretrained(model_name)
        self.model.eval()

    def score_words(self, sentence):
        input_ids = torch.tensor([self.tokenizer.encode(sentence, max_length=512, padding='max_length', add_special_tokens=True, truncation=True)])
        # input_ids = torch.tensor([self.tokenizer.encode(sentence)])
        tokenized_text = self.tokenizer.convert_ids_to_tokens(input_ids[0])

        with torch.no_grad():
            outputs = self.model(input_ids=input_ids)

        embeddings = outputs.last_hidden_state.squeeze(0)
        mean_embedding = embeddings.mean(dim=0)

        original_words = []
        original_word_embeddings = []
        current_word = ""
        prev_word = ""
        prev_embedding = ""
        current_embedding_lst = []

        for i, embedding in enumerate(embeddings):

            token = tokenized_text[i]
            if token in stop_wrds:
              continue
            elif token.startswith("##"):
                current_word = prev_word + token[2:]
                current_embedding_lst.append(prev_embedding)
                current_embedding_lst.append(embedding)
                current_embedding = sum(current_embedding_lst)/len(current_embedding_lst)

                original_words.pop()
                original_word_embeddings.pop()

                original_words.append(current_word)
                original_word_embeddings.append(current_embedding)

                prev_word = current_word
                prev_embedding = current_embedding
                current_embedding_lst = []
            else:
                original_words.append(token)
                original_word_embeddings.append(embedding)
                prev_word = token
                prev_embedding = embedding

        # for i, (token, embedding) in enumerate(zip(original_words, original_word_embeddings)):
        #   print(token)
        #   print(embedding.shape)

        word_scores = []
        new_scores = []
        for i in range(len(original_word_embeddings)):
            cos_sim = torch.nn.functional.cosine_similarity(original_word_embeddings[i], mean_embedding, dim=0)
            word_scores.append((original_words[i], cos_sim.item()))
            new_scores.append((original_words[i], original_word_embeddings[i], cos_sim.item()))

        return word_scores, mean_embedding, new_scores

    def print_top_values(self, data):
        top_values = int(len(data) * 0.6)
        if top_values < 10:
            top_values = int(len(data) * 0.7)
        if top_values < 4:
            top_values = int(len(data) * 0.8)

        finalLst = []
        for i in range(top_values):
            finalLst.append(data[i][0])

        return finalLst

    def keysOfSentence(self, sentence):
        final = []
        word_scores, mean_embedding, new_scores = self.score_words(sentence)
        word_scores.sort(key = lambda x: x[1], reverse=True)
        new_scores.sort(key = lambda x: x[2], reverse=True)

        finalKeysLst = self.print_top_values(word_scores)
        finalKeys = finalKeysLst

        return finalKeys, mean_embedding, new_scores

    def extract_keywords(self, text):
        self.load_model()
        return self.keysOfSentence(text)

def keysOfSentence(text):
    extractor = KeywordExtractor()

    # Load the model and tokenizer
    extractor.load_model()

    # Use GPU if available
    if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    extractor.model.to(device)
    extractor.model.eval()

    keywords, mean_embedding, new_scores = extractor.extract_keywords(text)
    return keywords

In [ ]:
text = 'এরপর গ্রেপ্তার করা হয় আরেক শিক্ষার্থীকে।'
keysOfSentence(text)

## Data Load and Keywords Generation

In [ ]:
import pandas as pd
df = pd.read_csv("./Data/test/testData1M.csv")
df = df.head(1000)
df.drop(columns=['keywords'], inplace=True)
df

,text
0,"উন্মুক্ত স্থান, বনাঞ্চল, জলাশয়ের পরিবেশ বিনষ্ট..."
1,দস্যু দমনে মেঘনায় একটি বাহিনীর একাধিক দলের টহ...
2,"এখন এসব নিয়ে প্রশ্ন হচ্ছে, তখন হারের ব্যাখ্যা ..."
3,গাজীপুরের কালিয়াকৈরের দিক থেকে আসা ওই বাসে আরও...
4,এতে জেলার ৪১ হাজারের বেশি জেলে কর্মহীন হয়ে পড়ে...
...,...
995,তবে আশির দশকে নির্মিত ছয়তলা এই বাড়িটি ‘জাহাজ ব...
996,তাঁকে মঞ্চে আহ্বান করেন রিয়েল এস্টেট ইনভেস্টর ...
997,এর আগে ভারতের রাজ্যগুলোর প্রতি দেওয়া এক পরামর্...
998,এ ক্ষেত্রে অরবিন্দ কেজরিওয়াল কিছু নাগরিক সুবিধ...


In [ ]:
df['text'][0]

'উন্মুক্ত স্থান, বনাঞ্চল, জলাশয়ের পরিবেশ বিনষ্ট হয়, এমন কাজ কখনো করব না।'

In [ ]:
rows = df.shape[0]
genDf = pd.DataFrame(columns = ["text", "bnKey2text", "bnKey2text-BERT-BASE", "Text Rank", "LIAAD/yake"])
for i in range(rows):
    text = df['text'][i]
    genDf.loc[i] = [text, keysOfSentence(text), keysOfSentenceBERTBASE(text), textrank(text), LIAADyake(text)]
    print(i, '  ', text)
print("Done..........!")

In [ ]:
genDf

In [ ]:
genDf.to_csv('./Data/test/ExtractedDifferentKeywords.csv', index=False)

## Keyword Performance Test

In [ ]:
import pandas as pd

df = pd.read_csv("./Data/test/ExtractedDifferentKeywords.csv")
df

,text,Human Keywords,bnKey2text,bnKey2text-BERT-BASE,Text Rank,LIAAD/yake
0,"উন্মুক্ত স্থান, বনাঞ্চল, জলাশয়ের পরিবেশ বিনষ্ট...","['বনাঞ্চল', 'জলাশয়ের', 'স্থান', 'উন্মুক্ত', '...","['বনাঞ্চল', 'জলাশয়ের', 'স্থান', 'উন্মুক্ত', '...","['বনাঞচল', 'কখনো', 'জলাশযের', 'সথান', 'বিনষট'...","['না', 'এমন', 'জলাশয়ের', 'পরিবেশ', 'কাজ', 'উন্...","['বনাঞ্চল', 'উন্মুক্ত', 'স্থান', 'জলাশয়ের', 'এ..."
1,দস্যু দমনে মেঘনায় একটি বাহিনীর একাধিক দলের টহ...,"[দলের, মেঘনায়, দমনে, একটি, দস্যু, বাহিনীর]","['মেঘনায়', 'বাহিনীর', 'দলের', 'একটি', 'দমনে',...","['মেঘনায', 'দসয', 'বাহিনীর', 'একাধিক', 'দলের',...","['দমনে', 'বাহিনীর', 'দলের', 'একটি', 'দস্যু']","['দস্যু', 'দরকার', 'দমনে', 'মেঘনায়', 'একটি', ..."
2,"এখন এসব নিয়ে প্রশ্ন হচ্ছে, তখন হারের ব্যাখ্যা ...","[এসব, প্রশ্ন, এখন, তখন, হারের, প্রশ্ন, ব্যাখ্যা]","['প্রশ্ন', 'প্রশ্ন', 'এখন', 'তখন', 'ব্যাখ্যা',...","['হচছে', 'নিযে', 'পরশন', 'নিযেই', 'বযাখযা', 'প...","['এখন', 'প্রশ্ন', 'প্রশ্ন', 'নিয়েই', 'নিয়ে', '...","['এখন', 'হচ্ছে', 'তখন', 'হতো', 'প্রশ্ন', 'এসব'..."
3,গাজীপুরের কালিয়াকৈরের দিক থেকে আসা ওই বাসে আরও...,"['কালিয়াকৈরের', 'অপহরণের', 'টাকা', 'দিক', 'থে...","['কালিয়াকৈরের', 'অপহরণের', 'টাকা', 'দিক', 'থে...","['মঠোফোন', 'ছিনিযে', 'দরবততরা', 'কালিযাকৈরের...","['থেকে', 'থেকে', 'দুটি', 'দুর্বৃত্তরা', '১০', ...","['গাজীপুরের', 'আরও', 'ছিল', 'এদের', 'থাকা', 'হ..."
4,এতে জেলার ৪১ হাজারের বেশি জেলে কর্মহীন হয়ে পড়ে...,"[হয়ে, জেলার, হাজারের, জেলে, কর্মহীন]","['জেলার', 'জেলে', 'হাজারের', 'কর্মহীন', 'হয়ে'...","['হযে', 'পডেছেন', 'করমহীন', 'হাজারের', 'জেলে']","['এতে', 'হয়ে', 'পড়েছেন', 'জেলে', 'জেলার']","['এতে', 'জেলার', 'হাজারের', 'পড়েছেন', 'বেশি', ..."
...,...,...,...,...,...,...
995,তবে আশির দশকে নির্মিত ছয়তলা এই বাড়িটি ‘জাহাজ ব...,"[ছয়তলা, আশির, বিল্ডিং, এই, নামে, বাড়িটি, জাহাজ]","['বিল্ডিং', 'জাহাজ', 'নামে', 'এই', 'আশির', 'ছয...","['ছযতলা', 'বাডিটি', 'বিলডিং', 'নিরমিত', 'এই', ...","['বাড়িটি', 'ছয়তলা', 'নির্মিত', 'তবে', 'দশকে', ...","['তবে', 'বিল্ডিং', 'নামে', 'পরিচিত', 'আশির', '..."
996,তাঁকে মঞ্চে আহ্বান করেন রিয়েল এস্টেট ইনভেস্টর ...,"['এস্টেট', 'ইনভেস্টর', 'আনোয়ার', 'মোহাম্মদ', ...","['এস্টেট', 'ইনভেস্টর', 'আনোয়ার', 'মোহাম্মদ', ...","['রিযেল', 'মোহামমদ', 'মঞচে', 'হোসেন', 'আনোয...","['তাঁকে', 'রিয়েল', 'আনোয়ার', 'মোহাম্মদ', 'মঞ্চ...","['তাঁকে', 'হোসেন', 'মঞ্চে', 'আহ্বান', 'করেন', ..."
997,এর আগে ভারতের রাজ্যগুলোর প্রতি দেওয়া এক পরামর্...,"[কাজ, আগে, জন্য, স্বরাষ্ট্র, বৈধভাবে, জাতীয়, ...","['রাজ্যগুলোর', 'মালয়েশিয়া', 'জন্য', 'মালয়েশ...","['মালযেশিযাতে', 'সবরাষটর', 'কেনদরীয', 'মনতরণাল...","['মালয়েশিয়াতে', 'মন্ত্রণালয়', 'দেওয়া', 'কেন্দ্...","['মালয়েশিয়া', 'রোহিঙ্গারা', 'বলেছিল', '‘বোঝা',..."
998,এ ক্ষেত্রে অরবিন্দ কেজরিওয়াল কিছু নাগরিক সুবিধ...,"[দেওয়ার, অরবিন্দ, ভর্তুকি, ক্ষেত্রে, মানোন্নয...","['অরবিন্দ', 'কেজরিওয়াল', 'কিছু', 'বাস্তবায়ন'...","['কেজরিওযাল', 'অরবিনদ', 'মাধযমে', 'কিছটা', 'পর...","['বাস্তবায়ন', 'দেওয়ার', 'কেজরিওয়াল', 'এ', 'কিছ...","['(বিদ্যুত্', 'পানিতে', 'ভর্তুকি', 'শিক্ষার', ..."


In [ ]:
import numpy as np
import pandas as pd

def mean_reciprocal_rank(predictions, actuals):
    for i, prediction in enumerate(predictions):
        if prediction in actuals:
            return 1 / (i + 1)
    return 0

def average_precision(predictions, actuals):
    precision_sum = 0
    num_correct = 0
    for i, prediction in enumerate(predictions):
        if prediction in actuals:
            num_correct += 1
            precision_sum += num_correct / (i + 1)
    if num_correct == 0:
        return 0
    return precision_sum / num_correct

def discounted_cumulative_gain(predictions, actuals):
    dcg = 0
    for i, prediction in enumerate(predictions):
        if prediction in actuals:
            gain = 1  # Binary relevance, 1 for a relevant item
            dcg += gain / np.log2(i + 2)  # i+2 because indexing starts from 0
    return dcg

def normalized_discounted_cumulative_gain(predictions, actuals):
    dcg = discounted_cumulative_gain(predictions, actuals)
    ideal_dcg = discounted_cumulative_gain(actuals, actuals)
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else 0
    return ndcg

In [ ]:
def keysTest(df, genKey, humanKey):

    # Calculate the metrics for each row in the DataFrame
    mrr_scores = []
    map_scores = []
    ndcg_scores = []

    for index, row in df.iterrows():
        predicted_keywords = [item for item in row[genKey] if item != '']  # Assuming keywords are separated by a comma and a space
        actual_keywords = [item for item in row[humanKey] if item != '']  # Assuming keywords are separated by a comma and a space

        mrr = mean_reciprocal_rank(predicted_keywords, actual_keywords)
        map_score = average_precision(predicted_keywords, actual_keywords)
        ndcg_score = normalized_discounted_cumulative_gain(predicted_keywords, actual_keywords)

        mrr_scores.append(mrr)
        map_scores.append(map_score)
        ndcg_scores.append(ndcg_score)

    # Calculate the average scores for the entire dataset
    average_mrr = np.mean(mrr_scores)
    average_map = np.mean(map_scores)
    average_ndcg = np.mean(ndcg_scores)

    return {'Column': genKey,
            'average_mrr': average_mrr,
            'average_map': average_map,
            'average_ndcg': average_ndcg
        }

In [ ]:
keysTest(df, 'bnKey2text', 'Human Keywords')

{'Column': 'bnKey2text',
 'average_mrr': 0.33586175493895853,
 'average_map': 0.3356251535258273,
 'average_ndcg': 0.33806783345515556}

In [ ]:
keysTest(df, 'bnKey2text-BERT-BASE', 'Human Keywords')

{'Column': 'bnKey2text-BERT-BASE',
 'average_mrr': 0.33485641967979113,
 'average_map': 0.3242755973229399,
 'average_ndcg': 0.31112850885929955}

In [ ]:
keysTest(df, 'Text Rank', 'Human Keywords')

{'Column': 'Text Rank',
 'average_mrr': 0.3380474769664079,
 'average_map': 0.3159566103352081,
 'average_ndcg': 0.2700835432227917}

In [ ]:
keysTest(df, 'LIAAD/yake', 'Human Keywords')

{'Column': 'LIAAD/yake',
 'average_mrr': 0.33552463372398256,
 'average_map': 0.31627677928532394,
 'average_ndcg': 0.3790937965831529}

In [ ]:
import pandas as pd

df = pd.read_csv("./Data/test/ExtractedDifferentKeywords.csv")


# Function to calculate exact matching percentage between two lists
def matching_percentage(list1, list2):
    if set(list1) == set(list2):
        return 100
    else:
        return 0

# Calculate matching percentage for each row
df['Matching Percentage'] = df.apply(lambda row: matching_percentage(row['Human Keywords'], row['bnKey2text']), axis=1)

# Calculate the average matching percentage
average_matching_percentage = df['Matching Percentage'].mean()

print("Average Matching Percentage:", average_matching_percentage)

Average Matching Percentage: 74.1
